In [1]:
import os
from typing import List, Optional, Sequence

import cv2
import numpy as np
import pandas as pd
import skimage
import torch

import byotrack
import byotrack.api.features_extractor
import byotrack.dataset.ctc as ctc_data
from byotrack.implementation.optical_flow.skimage import SkimageOpticalFlow
from byotrack.implementation.optical_flow.opencv import OpenCVOpticalFlow
from byotrack.implementation.linker.frame_by_frame.koft import (
    KOFTLinker,
    KOFTLinkerParameters,
)
from byotrack.implementation.linker.frame_by_frame.koft import (
    KalmanLinker,
    KalmanLinkerParameters,
)

/home/ddon0001/miniconda3/envs/koft/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# code reproduced from CTC submission link.py script

def get_average_size(detections_sequence: List[byotrack.Detections]) -> float:
    """Get the average size of cells in the dataset"""

    total_size = 0
    count = 0
    for detections in detections_sequence:
        if len(detections) <= 0:
            continue

        count += len(detections)
        total_size += int(detections.mass.sum().item())

    return total_size / (count + (count == 0))


def get_average_min_dist(detections_sequence: List[byotrack.Detections]) -> float:
    """Get the average minimal distance between cells in the dataset"""
    sum_min_dist = 0.0
    count = 0
    for detections in detections_sequence:
        if len(detections) <= 1:
            continue

        count += 1
        sum_min_dist += torch.cdist(detections.position, detections.position).sort(dim=1).values[:, 1].median().item()

    return sum_min_dist / (count + (count == 0))


def link(video: byotrack.Video, detections_sequence: Sequence[byotrack.Detections], **kwargs) -> List[byotrack.Track]:
    if kwargs["linker"] == "SKT":  # Simple Kalman Tracking from KOFT paper
        specs = KalmanLinkerParameters(
            association_threshold=kwargs["association_threshold"],  # Greedy is good
            detection_std=kwargs["detection_std"],  # Detections precisions. In CTC they are quite precise.
            process_std=kwargs["process_std"],  # ~ size of unmodeled displacement
            kalman_order=kwargs[
                "kalman_order"
            ],  # Order of the kalman filter (0: Brownian, 1: Directed, 2: Accelerated, ...)
            cost="euclidean",
            association_method="sparse_opt_smooth",  # Sparse linking (faster)
            n_valid=1,  # No spurious detections
            n_gap=kwargs["n_gap"],  # Few missing detections
            anisotropy=(kwargs["anisotropy"], 1.0, 1.0),  # For 3D anisotrope datasets
            split_factor=kwargs["split_factor"],  # Allows splits
            track_building="smoothed",  # RTS smoothing
        )

        linker = KalmanLinker(specs)

        # No need to read the video here as we don't use it for optflow
        return linker.run(
            [np.zeros((1, 1, 1, 1)) for _ in range(len(detections_sequence))],
            detections_sequence,
        )

    # KOFT
    if video.ndim == 5:  # 3D, but tvl1 is quite slow and unprecise in CTC
        optflow = SkimageOpticalFlow(
            skimage.registration.optical_flow_tvl1,
            downscale=4,
            parameters={"attachment": kwargs["attachment"]},
        )
    else:
        optflow = OpenCVOpticalFlow(cv2.FarnebackOpticalFlow.create(winSize=kwargs["win_size"]), downscale=4)

    specs = KOFTLinkerParameters(
        association_threshold=kwargs["association_threshold"],  # Greedy is good
        detection_std=kwargs["detection_std"],  # Detections precisions. In CTC they are quite precise.
        process_std=kwargs["process_std"],  #  ~size of unmodeled displacement
        flow_std=kwargs["flow_std"],  # ~ Optical flow errors (quite low performances in CTC)
        kalman_order=kwargs["kalman_order"],  # Order of the kalman filter (1: Directed, 2: Accelerated, ...)
        cost="euclidean",
        association_method="sparse_opt_smooth",  # Sparse linking (faster)
        n_valid=1,  # No spurious detections
        n_gap=kwargs["n_gap"],  # Few missing detections
        anisotropy=(kwargs["anisotropy"], 1.0, 1.0),  # For 3D anisotrope datasets
        split_factor=kwargs["split_factor"],  # Allows splits
        track_building="smoothed",  # RTS smoothing
    )

    linker = KOFTLinker(specs, optflow)

    return linker.run(video, detections_sequence)

def track(
    data_path: str,
    seg_path: str,
    out_path: str,
    *,
    linker: Optional[str] = None,
    association_threshold=0.0,
    detection_std=0.0,
    process_std=0.0,
    flow_std=0.0,
    kalman_order=0,
    anisotropy=0.0,
    split_factor=-1.0,
    n_gap=1,
    win_size=0,
    attachment=10.0,
    detections_sequence=None,
):
    # Load the video and normalize it
    video = byotrack.Video(data_path)  # Load videos
    video.set_transform(
        byotrack.VideoTransformConfig(
            aggregate=True,
            normalize=True,
            compute_stats_on=50 if video.ndim == 4 else 10,
        )
    )
    n_digit=len(str(len(video)))+1

    # Load segmentations
    if not detections_sequence:
        detections_sequence = ctc_data.GroundTruthDetector().run(byotrack.Video(seg_path))

    # Set default parameters
    if video.ndim == 4:  # 2D
        anisotropy = 1.0

    # Anisotropy is computed if not given based on detections (Depends on direciton)
    if video.ndim == 5 and anisotropy <= 0.0:  # 3D
        sizes = sum(
            detections.bbox[:, detections.dim :].to(torch.float32).mean(axis=0) for detections in detections_sequence
        ) / len(detections_sequence)
        anisotropy = float(sizes[1:].mean() / sizes[0])

    # Useful features for setting the parameters
    spot_size = get_average_size(detections_sequence)
    closest_spot_dist = get_average_min_dist(detections_sequence)
    cell_increase = (len(detections_sequence[-1]) - len(detections_sequence[0])) / len(detections_sequence[0])

    if video.ndim == 5:  # 3D + T + C
        spot_radius = float((spot_size * anisotropy * 3 / 4 / np.pi) ** (1 / 3))  # area = 4/3 pi R^3 / ani
    else:
        spot_radius = float(np.sqrt(spot_size / np.pi))  # pi R^2

    print("===========Dataset features============")
    print("Spot radius: ", spot_radius)
    print("Closest spot dist:", closest_spot_dist)
    print("Anisotropy of the Z axis:", anisotropy)
    print("Cell increase between first and last frame:", cell_increase * 100, "%")

    # For farneback, we set by default the winsize ~= cell diameter (after a downscale of 4)
    if win_size == 0:
        win_size = max(10, int(spot_radius / 2))

    # Detections std ~= 1/2 cell radius
    if detection_std <= 0.0:
        detection_std = spot_radius / 2

    # Process_std ~= 3 spot radius
    if process_std <= 0.0:
        process_std = 3 * spot_radius

    # flow_std = process_std (noisy flow that we trust as much as our process)
    if flow_std <= 0.0:
        flow_std = process_std

    if video.ndim == 5 and anisotropy != 1.0:  # Anisotrope 3D, we scale errors on the Z axis
        detection_std = torch.tensor(
            (detection_std / anisotropy, detection_std, detection_std),
            dtype=torch.float32,
        )
        process_std = torch.tensor((process_std / anisotropy, process_std, process_std), dtype=torch.float32)
        flow_std = torch.tensor(
            (flow_std, flow_std, flow_std), dtype=torch.float32
        )  # The flow has no reason to be scaled

    if split_factor < 0:
        if (
            cell_increase > 0.3 and cell_increase * len(detections_sequence[0]) > 1
        ):  # At least 30% of augmentation of cells to activate mitose
            split_factor = 1.0
        else:
            split_factor = 0.0

    # For 3D videos, we could do a conversion 3D to 2D for simple 3D (cf conversion scripts, but it is complex
    # and useful only for computational time)
    # Or use a 3D optical flow, but skimage TVL1 is quite expensive and not very accurate in CTC
    # By default, we simply use KalmanLinker in 3D (much faster and still has very good results)
    linker = linker if linker is not None else ("KOFT" if video.ndim == 4 else "SKT")

    # For the association threshold we just set at 3 times the spot radius
    association_threshold = (
        association_threshold if association_threshold > 0.0 else max(spot_radius * 3, closest_spot_dist)
    )

    parameters = {
        "linker": linker,
        "association_threshold": association_threshold,
        "detection_std": detection_std,
        "process_std": process_std,
        "flow_std": flow_std,
        "kalman_order": kalman_order,
        "split_factor": split_factor,
        "anisotropy": anisotropy,
        "n_gap": n_gap,
        "win_size": win_size,
        "attachment": attachment,
    }

    print("==============parameters================")
    print(parameters)

    print("===============Running==================")
    # Let's track!
    tracks = link(video, detections_sequence, **parameters)
    print(f"Produced {len(tracks)} tracks.")

    ctc_data.save_tracks(
        os.path.join(out_path, 'RES/'),
        tracks,
        detections_sequence=detections_sequence,
        default_radius=spot_radius * 3,
        shape=video.shape[1:],
        n_digit=n_digit,
        anisotropy=anisotropy,
    )

In [3]:
root_dir = '/home/ddon0001/PhD/data/trackastra_training/draga'

summary_df = pd.read_csv('/home/ddon0001/PhD/experiments/trackastra_training_tracktour/ds_summary.csv')

out_root = '/home/ddon0001/PhD/experiments/trackastra_training_koft_ctc/'

In [ ]:
errored = {}
for row in summary_df.itertuples():
    ds_name = row.ds_name
    im_path = row.im_path
    seg_path = row.seg_path
    out_path = os.path.join(out_root, ds_name)

    if os.path.exists(out_path):
        print(f"Skipping {ds_name} as output path already exists.")
        continue

    try:
        track(im_path, seg_path, out_path)
    except Exception as e:
        print(f"Error processing {ds_name}: {e}")
        errored[ds_name] = e

In [10]:
errored

{'celegans_dispim_nih_diSPIM_deconv_1': RuntimeError('The expanded size of the tensor (2) must match the existing size (3) at non-singleton dimension 0.  Target sizes: [2].  Tensor sizes: [3]'),
 'mskcc-confocal_mskcc_confocal_s3_370_isotropic': RuntimeError('The expanded size of the tensor (2) must match the existing size (3) at non-singleton dimension 0.  Target sizes: [2].  Tensor sizes: [3]'),
 'mskcc-confocal_mskcc_confocal_s1_400_isotropic': RuntimeError('The expanded size of the tensor (2) must match the existing size (3) at non-singleton dimension 0.  Target sizes: [2].  Tensor sizes: [3]'),
 'mskcc-confocal_mskcc_confocal_s2_376_isotropic': RuntimeError('The expanded size of the tensor (2) must match the existing size (3) at non-singleton dimension 0.  Target sizes: [2].  Tensor sizes: [3]'),
 'epithelia_per01': TypeError("can't convert np.ndarray of type numpy.ulonglong. The only supported types are: float64, float32, float16, complex64, complex128, int64, int32, int16, int8,

In [7]:

from traccuracy.loaders import load_ctc_data
from traccuracy.matchers import CTCMatcher
from traccuracy.metrics import CTCMetrics
from traccuracy.utils import export_graphs_to_geff

for row in summary_df.itertuples():
    if row.ds_name in errored:
        continue
    gt_path = row.tra_gt_path
    res_path = os.path.join(out_root, row.ds_name, 'RES/')
    out_metrics_path = os.path.join(out_root, row.ds_name, 'matched_solution.zarr')
    if os.path.exists(out_metrics_path):
        print(f"Skipping {row.ds_name} as metrics already computed.")
        continue

    print(f"Processing {row.ds_name}...")
    gt_data = load_ctc_data(gt_path)
    res_data = load_ctc_data(res_path)
    try:
        matcher = CTCMatcher()
        matched = matcher.compute_mapping(gt_data, res_data)
        metrics = CTCMetrics()
        results = metrics.compute(matched)
        export_graphs_to_geff(out_metrics_path, matched, [results])
    except ValueError as e:
        errored[row.ds_name] = e

Skipping deepcell_train-80 as metrics already computed.
Skipping deepcell_train-25 as metrics already computed.
Skipping deepcell_train-62 as metrics already computed.
Skipping deepcell_train-61 as metrics already computed.
Skipping deepcell_train-23 as metrics already computed.
Skipping deepcell_train-59 as metrics already computed.
Skipping deepcell_train-82 as metrics already computed.
Skipping deepcell_train-03 as metrics already computed.
Skipping deepcell_train-36 as metrics already computed.
Skipping deepcell_train-45 as metrics already computed.
Skipping deepcell_train-85 as metrics already computed.
Skipping deepcell_train-43 as metrics already computed.
Skipping deepcell_train-14 as metrics already computed.
Skipping deepcell_train-31 as metrics already computed.
Skipping deepcell_train-24 as metrics already computed.
Skipping deepcell_train-81 as metrics already computed.
Skipping deepcell_train-32 as metrics already computed.
Skipping deepcell_train-64 as metrics already co

Evaluating FN edges: 100%|██████████| 47217/47217 [00:00<00:00, 449757.12it/s]


Skipping trackmate_TCells-01 as metrics already computed.
Skipping trackmate_NMeningitidis-01 as metrics already computed.
Skipping ker_phasecontrast_dataset2-sub_5-exp1_F0018 as metrics already computed.
Skipping ker_phasecontrast_dataset2-sub_5-exp1_F0001 as metrics already computed.
Skipping ker_phasecontrast_dataset2-sub_5-exp1_F0004 as metrics already computed.
Skipping ker_phasecontrast_dataset2-sub_5-exp1_F0015 as metrics already computed.
Skipping ker_phasecontrast_dataset2-sub_5-exp1_F0002 as metrics already computed.
Skipping ker_phasecontrast_dataset1-sub_5-exp1_F0003 as metrics already computed.
Skipping ker_phasecontrast_dataset1-sub_5-exp1_F0008 as metrics already computed.
Skipping ker_phasecontrast_dataset1-sub_5-exp1_F0016 as metrics already computed.
Skipping ker_phasecontrast_dataset1-sub_5-exp1_F0014 as metrics already computed.
Skipping ker_phasecontrast_dataset1-sub_5-exp1_F0015 as metrics already computed.
Skipping ker_phasecontrast_dataset1-sub_5-exp1_F0002 as m

Evaluating FN edges: 100%|██████████| 111173/111173 [00:00<00:00, 470466.87it/s]


Processing mskcc-confocal_mskcc_confocal_s1_400_isotropic...


Evaluating FN edges: 100%|██████████| 129877/129877 [00:00<00:00, 453779.45it/s]


Processing mskcc-confocal_mskcc_confocal_s2_376_isotropic...


Evaluating FN edges: 100%|██████████| 114935/114935 [00:00<00:00, 477174.61it/s]


Skipping epithelia_per01 as metrics already computed.
Skipping epithelia_per03 as metrics already computed.
Skipping epithelia_per02 as metrics already computed.
Skipping vanvliet_pheA-150324-03 as metrics already computed.
Skipping vanvliet_pheA-160112-06 as metrics already computed.
Skipping vanvliet_pheA-150325-04 as metrics already computed.
Skipping vanvliet_pheA-150324-05 as metrics already computed.
Skipping vanvliet_pheA-160112-04 as metrics already computed.
Skipping vanvliet_cib-140408-04 as metrics already computed.
Skipping vanvliet_cib-140408-02 as metrics already computed.
Skipping vanvliet_cib-140415-08 as metrics already computed.
Skipping vanvliet_cib-140409-03 as metrics already computed.
Skipping vanvliet_cib-140415-13 as metrics already computed.
Skipping vanvliet_cib-140408-10 as metrics already computed.
Skipping vanvliet_metA-150318-06 as metrics already computed.
Skipping vanvliet_metA-150331-12 as metrics already computed.
Skipping vanvliet_metA-151222-11 as me

In [6]:
errored = {}

In [5]:
# trying to fix errored datasets

celegans = summary_df[summary_df.ds_name.str.contains('mskcc-confocal_mskcc_confocal')]
for row in celegans.itertuples():
    ds_name = row.ds_name
    im_path = row.im_path
    seg_path = row.seg_path
    out_path = os.path.join(out_root, ds_name)

    track(im_path, seg_path, out_path)


Detections (Load from CTC format): 100%|██████████| 370/370 [00:05<00:00, 73.54it/s]


===========Dataset features============
Spot radius:  2.319049853256222
Closest spot dist: 9.186360512553035
Anisotropy of the Z axis: 1.6508742570877075
Cell increase between first and last frame: 29850.0 %
==============parameters================
{'linker': 'SKT', 'association_threshold': 9.186360512553035, 'detection_std': tensor([0.7024, 1.1595, 1.1595]), 'process_std': tensor([4.2142, 6.9571, 6.9571]), 'flow_std': tensor([6.9571, 6.9571, 6.9571]), 'kalman_order': 0, 'split_factor': 1.0, 'anisotropy': 1.6508742570877075, 'n_gap': 1, 'win_size': 10, 'attachment': 10.0}
===============Running==================


Kalman filter linking: 100%|██████████| 370/370 [00:06<00:00, 57.52it/s] 


Produced 957 tracks.


Detections (Load from CTC format): 100%|██████████| 400/400 [00:05<00:00, 78.92it/s]


...

Detections (Load from CTC format): 100%|██████████| 376/376 [00:04<00:00, 79.13it/s]


===========Dataset features============
Spot radius:  2.3807798076227398
Closest spot dist: 9.600279811848985
Anisotropy of the Z axis: 1.6348716020584106
Cell increase between first and last frame: 30350.0 %
==============parameters================
{'linker': 'SKT', 'association_threshold': 9.600279811848985, 'detection_std': tensor([0.7281, 1.1904, 1.1904]), 'process_std': tensor([4.3687, 7.1423, 7.1423]), 'flow_std': tensor([7.1423, 7.1423, 7.1423]), 'kalman_order': 0, 'split_factor': 1.0, 'anisotropy': 1.6348716020584106, 'n_gap': 1, 'win_size': 10, 'attachment': 10.0}
===============Running==================


Kalman filter linking: 100%|██████████| 376/376 [00:06<00:00, 56.60it/s] 


Produced 948 tracks.


Saving tracks to CTC: 100%|██████████| 376/376 [00:13<00:00, 28.76it/s]
